In [1]:
pip install ipympl opencv-python

Note: you may need to restart the kernel to use updated packages.


In [1]:
%matplotlib widget

In [ ]:
import cv2

# =============================
# PATHS
# =============================
IMG_PATH = r"D:\apollo_newmachinedata\TEMPLATE_RIGHT\temp\patches_rtor1\380__r007_c000.png"
SAVE_IMG = r"D:\apollo_newmachinedata\TEMPLATE_RIGHT\temp\roi.png"

# =============================
# LOAD IMAGE
# =============================
img = cv2.imread(IMG_PATH)

if img is None:
    raise ValueError("Image not found")

H, W = img.shape[:2]
print("Image shape:", (H, W))

# =============================
# VIEW SETTINGS
# =============================
win_h, win_w = 800, 800
x_off, y_off = 0, 0
zoom = 1.0

drawing = False
x1 = y1 = x2 = y2 = 0

# =============================
# MOUSE FUNCTION (AUTO SAVE)
# =============================
def draw_roi(event, x, y, flags, param):
    global drawing, x1, y1, x2, y2

    ix = int(x / zoom + x_off)
    iy = int(y / zoom + y_off)

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        x1, y1 = ix, iy

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        x2, y2 = ix, iy

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        x2, y2 = ix, iy

        # =============================
        # AUTO SAVE ROI HERE
        # =============================
        if x1 != x2 and y1 != y2:
            x_min, x_max = min(x1, x2), max(x1, x2)
            y_min, y_max = min(y1, y2), max(y1, y2)

            roi = img[y_min:y_max, x_min:x_max]

            cv2.imwrite(SAVE_IMG, roi)

            print("✅ ROI saved:", SAVE_IMG)
            print("ROI shape:", roi.shape)
        else:
            print("❌ Invalid ROI")

# =============================
# WINDOW
# =============================
cv2.namedWindow("Viewer", cv2.WINDOW_NORMAL)
cv2.setMouseCallback("Viewer", draw_roi)

print("\nControls:")
print("W/A/S/D → move")
print("+ / -   → zoom")
print("Mouse drag → ROI (auto-saves on release)")
print("Q → quit\n")

# =============================
# MAIN LOOP
# =============================
while True:

    # keep offsets valid
    x_off = max(0, min(x_off, W - win_w))
    y_off = max(0, min(y_off, H - win_h))

    view = img[y_off:y_off+win_h, x_off:x_off+win_w]
    display = cv2.resize(view, None, fx=zoom, fy=zoom)

    # draw rectangle live
    if drawing and x1 != x2 and y1 != y2:
        dx1 = int((x1 - x_off) * zoom)
        dy1 = int((y1 - y_off) * zoom)
        dx2 = int((x2 - x_off) * zoom)
        dy2 = int((y2 - y_off) * zoom)

        cv2.rectangle(display, (dx1, dy1), (dx2, dy2), (0,255,0), 2)

    cv2.imshow("Viewer", display)

    key = cv2.waitKey(1) & 0xFF

    # PAN
    if key == ord('w'): y_off -= 200
    elif key == ord('s'): y_off += 200
    elif key == ord('a'): x_off -= 200
    elif key == ord('d'): x_off += 200

    # ZOOM
    elif key == ord('+') or key == ord('='): zoom *= 1.2
    elif key == ord('-'): zoom /= 1.2

    elif key == ord('q'):
        break

cv2.destroyAllWindows()

Image shape: (4200, 4096)

Controls:
W/A/S/D → move
+ / -   → zoom
Mouse drag → ROI (auto-saves on release)
Q → quit

✅ ROI saved: D:\apollo_newmachinedata\TEMPLATE_RIGHT\temp\roi.png
ROI shape: (67, 13, 3)
✅ ROI saved: D:\apollo_newmachinedata\TEMPLATE_RIGHT\temp\roi.png
ROI shape: (451, 288, 3)


In [10]:
# gave 6 images/7
import cv2
import os
from glob import glob
import numpy as np

# =============================
# PATHS
# =============================
TEMPLATE_PATH = r"D:\apollo_newmachinedata\test\template.png"
INPUT_FOLDER = r"D:\apollo_newmachinedata\test\test_images\patches_rtor1"
OUTPUT_FOLDER = r"D:\apollo_newmachinedata\test\presentfull21"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =============================
# LOAD TEMPLATE
# =============================
template = cv2.imread(TEMPLATE_PATH, 0)
template = cv2.GaussianBlur(template, (5,5), 0)

th, tw = template.shape

# =============================
# SETTINGS
# =============================
THRESHOLD = 0.5  # lowered slightly
SCALES = [0.9, 1.0, 1.1]  # handle slight size variation

image_paths = glob(os.path.join(INPUT_FOLDER, "*.png"))

print("Total images:", len(image_paths))

for path in image_paths:

    img = cv2.imread(path, 0)
    if img is None:
        continue

    img_blur = cv2.GaussianBlur(img, (5,5), 0)

    best_score = 0

    # -------------------------
    # MULTI-SCALE MATCH
    # -------------------------
    for scale in SCALES:
        resized_template = cv2.resize(template, None, fx=scale, fy=scale)

        if resized_template.shape[0] > img.shape[0] or resized_template.shape[1] > img.shape[1]:
            continue

        result = cv2.matchTemplate(img_blur, resized_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = cv2.minMaxLoc(result)

        best_score = max(best_score, max_val)

    # -------------------------
    # DECISION
    # -------------------------
    if best_score >= THRESHOLD:
        print(f"✅ Found: {os.path.basename(path)} | Score: {best_score:.3f}")
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, os.path.basename(path)), img)
    else:
        print(f"❌ Not found: {os.path.basename(path)} | Score: {best_score:.3f}")

print("\n✅ Done")


Total images: 160
❌ Not found: 219__r000_c000.png | Score: 0.227
✅ Found: 219__r001_c000.png | Score: 0.941
❌ Not found: 219__r002_c000.png | Score: 0.207
❌ Not found: 219__r003_c000.png | Score: 0.215
❌ Not found: 219__r004_c000.png | Score: 0.225
❌ Not found: 219__r005_c000.png | Score: 0.244
❌ Not found: 219__r006_c000.png | Score: 0.205
❌ Not found: 219__r007_c000.png | Score: 0.263
❌ Not found: 219__r008_c000.png | Score: 0.197
❌ Not found: 219__r009_c000.png | Score: 0.206
✅ Found: 220__r000_c000.png | Score: 0.918
❌ Not found: 220__r001_c000.png | Score: 0.235
❌ Not found: 220__r002_c000.png | Score: 0.264
❌ Not found: 220__r003_c000.png | Score: 0.231
❌ Not found: 220__r004_c000.png | Score: 0.229
✅ Found: 220__r005_c000.png | Score: 0.885
❌ Not found: 220__r006_c000.png | Score: 0.333
❌ Not found: 220__r007_c000.png | Score: 0.230
❌ Not found: 220__r008_c000.png | Score: 0.235
❌ Not found: 220__r009_c000.png | Score: 0.226
✅ Found: 221__r000_c000.png | Score: 0.999
❌ Not found

In [11]:
import cv2
import os
from glob import glob
import numpy as np
import time

# =============================
# PATHS
# =============================
TEMPLATE_PATH = r"D:\apollo_newmachinedata\test\template.png"
INPUT_FOLDER = r"D:\apollo_newmachinedata\test\test_images\patches_rtor1"
OUTPUT_FOLDER = r"D:\apollo_newmachinedata\test\presentfull21"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =============================
# LOAD TEMPLATE
# =============================
template = cv2.imread(TEMPLATE_PATH, 0)
template = cv2.GaussianBlur(template, (5,5), 0)

th, tw = template.shape

# =============================
# SETTINGS
# =============================
THRESHOLD = 0.5
SCALES = [0.9, 1.0, 1.1]

image_paths = glob(os.path.join(INPUT_FOLDER, "*.png"))

print("Total images:", len(image_paths))

# =============================
# TOTAL TIMER START
# =============================
total_start = time.time()

# =============================
# PROCESS LOOP
# =============================
for path in image_paths:

    img_start = time.time()  # ⏱ per-image start

    img = cv2.imread(path, 0)
    if img is None:
        continue

    img_blur = cv2.GaussianBlur(img, (5,5), 0)

    best_score = 0

    # -------------------------
    # MULTI-SCALE MATCH
    # -------------------------
    for scale in SCALES:
        resized_template = cv2.resize(template, None, fx=scale, fy=scale)

        if resized_template.shape[0] > img.shape[0] or resized_template.shape[1] > img.shape[1]:
            continue

        result = cv2.matchTemplate(img_blur, resized_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = cv2.minMaxLoc(result)

        best_score = max(best_score, max_val)

    # -------------------------
    # DECISION
    # -------------------------
    if best_score >= THRESHOLD:
        print(f"✅ Found: {os.path.basename(path)} | Score: {best_score:.3f}")
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, os.path.basename(path)), img)
    else:
        print(f"❌ Not found: {os.path.basename(path)} | Score: {best_score:.3f}")

    # ⏱ per-image end
    img_time = time.time() - img_start
    print(f"⏱ Time for {os.path.basename(path)}: {img_time:.3f} sec")

# =============================
# TOTAL TIME
# =============================
total_time = time.time() - total_start
avg_time = total_time / len(image_paths) if image_paths else 0

print("\n=============================")
print(f"⏱ Total time: {total_time:.3f} sec")
print(f"⏱ Avg time per image: {avg_time:.3f} sec")
print("✅ Done")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Total images: 160
❌ Not found: 219__r000_c000.png | Score: 0.227
⏱ Time for 219__r000_c000.png: 3.032 sec
✅ Found: 219__r001_c000.png | Score: 0.941
⏱ Time for 219__r001_c000.png: 3.489 sec
❌ Not found: 219__r002_c000.png | Score: 0.207
⏱ Time for 219__r002_c000.png: 2.865 sec
❌ Not found: 219__r003_c000.png | Score: 0.215
⏱ Time for 219__r003_c000.png: 2.851 sec
❌ Not found: 219__r004_c000.png | Score: 0.225
⏱ Time for 219__r004_c000.png: 3.080 sec
❌ Not found: 219__r005_c000.png | Score: 0.244
⏱ Time for 219__r005_c000.png: 2.843 sec
❌ Not found: 219__r006_c000.png | Score: 0.205
⏱ Time for 219__r006_c000.png: 2.775 sec
❌ Not found: 219__r007_c000.png | Score: 0.263
⏱ Time for 219__r007_c000.png: 2.590 sec
❌ Not found: 219__r008_c000.png | Score: 0.197
⏱ Time for 219__r008_c000.png: 3.154 sec
❌ Not found: 219__r009_c000.png | Score: 0.206
⏱ Tim

In [ ]:
import cv2
import os
from glob import glob
import numpy as np
import time

TEMPLATE_PATH = r"D:\apollo_newmachinedata\Right_side (1)\Right_side\IMAGE\patches_rtor1\roi.png"
INPUT_FOLDER = r"D:\apollo_newmachinedata\Right_side (1)\Right_side"
OUTPUT_FOLDER = r"D:\apollo_newmachinedata\Right_side (1)\Right_side\IMAGE\patches_rtor1\output"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

template = cv2.imread(TEMPLATE_PATH, 0)
template = cv2.GaussianBlur(template, (5,5), 0)

THRESHOLD = 0.5
SCALES = [0.9, 1.0, 1.1]

# ✅ PRECOMPUTE SCALED TEMPLATES
scaled_templates = []
for scale in SCALES:
    t = cv2.resize(template, None, fx=scale, fy=scale)
    scaled_templates.append((t, t.shape))

image_paths = glob(os.path.join(INPUT_FOLDER, "*.png"))

print("Total images:", len(image_paths))

# local refs (micro-opt)
mt = cv2.matchTemplate
mml = cv2.minMaxLoc

total_start = time.time()

for path in image_paths:

    img = cv2.imread(path, 0)
    if img is None:
        continue

    img_blur = cv2.GaussianBlur(img, (5,5), 0)

    best_score = 0

    for resized_template, (th, tw) in scaled_templates:

        if th > img.shape[0] or tw > img.shape[1]:
            continue

        result = mt(img_blur, resized_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = mml(result)

        if max_val > best_score:
            best_score = max_val

        # ✅ early exit
        if best_score >= THRESHOLD:
            break

    if best_score >= THRESHOLD:
        print(f"✅ Found: {os.path.basename(path)} | Score: {best_score:.3f}")
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, os.path.basename(path)), img)
    else:
        print(f"❌ Not found: {os.path.basename(path)} | Score: {best_score:.3f}")

total_time = time.time() - total_start
print(f"\n⏱ Total time: {total_time:.3f} sec")
print("✅ Done")

Total images: 5
